# Browser-Based Real-Time Face Swap with Colab GPU

This notebook captures your webcam directly in the browser, processes frames on Colab's GPU, and displays the face-swapped result in real-time.

**No Windows client needed!** Everything runs in this notebook.

## Steps:
1. Select a GPU runtime (T4 recommended)
2. Install dependencies
3. Upload a source face image
4. Start webcam capture and processing

## 1. Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install -q opencv-python-headless insightface onnxruntime-gpu

## 2. Verify GPU

In [ ]:
import torch
import onnxruntime as ort

print("CUDA available:", torch.cuda.is_available())
print("ONNX Runtime providers:", ort.get_available_providers())

assert "CUDAExecutionProvider" in ort.get_available_providers(), "GPU not available!"

## 3. Download Face Swap Model

In [ ]:
import urllib.request
from pathlib import Path

MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(exist_ok=True)
SWAPPER_PATH = MODEL_DIR / "inswapper_128.onnx"

if not SWAPPER_PATH.exists():
    print("Downloading face swap model...")
    urllib.request.urlretrieve(
        "https://huggingface.co/deepinsight/inswapper/resolve/main/inswapper_128.onnx",
        SWAPPER_PATH,
    )
    print(f"Downloaded: {SWAPPER_PATH.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print(f"Model already exists: {SWAPPER_PATH}")

## 4. Initialize Face Processing

In [ ]:
import cv2
import insightface
import numpy as np

ORT_PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]

# Initialize face analyzer
print("Initializing face analyzer...")
FACE_ANALYSER = insightface.app.FaceAnalysis(name="buffalo_l", providers=ORT_PROVIDERS)
FACE_ANALYSER.prepare(ctx_id=0, det_size=(640, 640))

# Initialize face swapper
print("Initializing face swapper...")
FACE_SWAPPER = insightface.model_zoo.get_model(str(SWAPPER_PATH), providers=ORT_PROVIDERS)

print("✓ GPU face processing initialized")

def get_one_face(frame):
    """Detect the first face in the frame."""
    faces = FACE_ANALYSER.get(frame)
    if not faces:
        return None
    return min(faces, key=lambda face: face.bbox[0])

def swap_face(source_face, target_frame):
    """Swap source face into target frame."""
    target_face = get_one_face(target_frame)
    if target_face is None:
        return target_frame  # No face detected, return original
    
    result = FACE_SWAPPER.get(target_frame, target_face, source_face, paste_back=True)
    return result

## 5. Upload Source Face Image

Upload an image containing the face you want to swap onto your webcam.

In [ ]:
from google.colab import files
from IPython.display import Image, display
import io

print("Upload a source face image (JPG, PNG):")
uploaded = files.upload()

# Get the uploaded file
source_filename = list(uploaded.keys())[0]
source_bytes = uploaded[source_filename]

# Decode image
source_array = np.frombuffer(source_bytes, np.uint8)
SOURCE_IMAGE = cv2.imdecode(source_array, cv2.IMREAD_COLOR)

print(f"Source image loaded: {SOURCE_IMAGE.shape}")
display(Image(data=source_bytes, width=300))

# Extract source face
print("\nDetecting source face...")
SOURCE_FACE = get_one_face(SOURCE_IMAGE)
if SOURCE_FACE is None:
    raise ValueError("No face detected in source image!")
print("✓ Source face detected and cached")

## 6. Start Real-Time Face Swap

This will:
1. Capture your webcam in the browser
2. Send frames to Colab GPU for processing
3. Display the face-swapped result in real-time

**Click "Allow" when prompted for webcam access.**

Press **Stop** button to end the stream.

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image
import io
import time

def webcam_to_numpy(quality=0.8, size=(640, 480)):
    """Capture a frame from webcam and return as numpy array."""
    js = Javascript('''
    async function captureFrame(quality, width, height) {
      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: {width, height}});
      
      video.srcObject = stream;
      await video.play();
      
      // Wait for video to be ready
      await new Promise(resolve => setTimeout(resolve, 500));
      
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      
      stream.getTracks().forEach(track => track.stop());
      
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    
    data = eval_js(f'captureFrame({quality}, {size[0]}, {size[1]})')
    binary = b64decode(data.split(',')[1])
    
    img = PIL.Image.open(io.BytesIO(binary))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

print("Starting webcam... (this may take a moment)")
print("="*50)

# Test capture
test_frame = webcam_to_numpy(quality=0.8, size=(640, 480))
print(f"✓ Webcam ready! Resolution: {test_frame.shape[1]}x{test_frame.shape[0]}")
print("\nProcessing first frame...")

# Process and display
result = swap_face(SOURCE_FACE, test_frame)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
result_pil = PIL.Image.fromarray(result_rgb)

display(result_pil)
print("\n✓ Face swap working!")
print("\nRun the next cell for continuous streaming.")

## 7. Continuous Stream (Run this for live video)

**Note:** Due to Colab limitations, this captures frames repeatedly rather than true video streaming. Expect 2-5 FPS.

Press **Interrupt** (⏹️) to stop.

In [ ]:
from IPython.display import clear_output
import time

print("Starting continuous face swap stream...")
print("Press the STOP button (■) to end.\n")

frame_count = 0
start_time = time.time()

try:
    while True:
        # Capture frame
        frame = webcam_to_numpy(quality=0.7, size=(640, 480))
        
        # Process
        result = swap_face(SOURCE_FACE, frame)
        
        # Convert and display
        result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        result_pil = PIL.Image.fromarray(result_rgb)
        
        clear_output(wait=True)
        display(result_pil)
        
        frame_count += 1
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        
        print(f"Frames: {frame_count} | FPS: {fps:.1f} | Elapsed: {elapsed:.1f}s")
        print("Press STOP (■) to end stream")
        
except KeyboardInterrupt:
    print("\n✓ Stream stopped")
    print(f"Total frames: {frame_count}")
    print(f"Average FPS: {frame_count / (time.time() - start_time):.1f}")

## Notes

- **Performance**: Browser → Colab → Browser adds latency. Expect 2-5 FPS.
- **Privacy**: Your webcam data stays in your browser and Colab session. Nothing is stored.
- **GPU**: Uses Colab's T4 GPU for real-time face swapping.
- **Limitations**: This won't work in Zoom/Teams directly - it's just for preview.

For production use with virtual cameras, use the Windows client version instead.